# Phase 1 (awake reference): neurons, firing activity and ensembles

First step of the state ↔ ensembles program. Everything is defined on awake, stimulation-free seconds only
(65 region signals, 8 recordings, 7 mice), so it can be reused unchanged for isoflurane. Code: `src/ensembles.py`.

**Units.** Allen quality filters, then a rate floor (≥ 0.5 Hz awake) and removal of duplicate units (pairs ≤ 60 µm
apart whose spikes coincide within ±0.5 ms for ≥ 20 % of spikes and whose mean waveforms correlate ≥ 0.9).

**Firing descriptors per unit.** Rate; CV2 (spike-time irregularity, high = bursty); Fano factor of 1-s counts;
population coupling (r with the summed rate of all other units); waveform duration (< 0.4 ms = narrow-spiking,
putative fast-spiking interneuron); depth along the probe.

**Ensembles, three independent definitions; a neuron may belong to none.**
- *lfp*: each neuron's correlation with a motif's expression (original M1; tracker expression M1), tested against
  500 circular shifts (same shifts for every neuron), Benjamini–Hochberg across neurons → +, − or none. On raw
  rates and with the population mean removed.
- *getclust*: Mishler's algorithm on spiking alone; neurons in no cluster are unassigned.
- *corr*: correlations between neurons (population mean removed) that beat a null where every neuron is shifted
  independently (BH), grouped into communities by modularity (Newman's leading-eigenvector method); communities
  of ≥ 3 neurons are ensembles.

**Validation.** Split-half reliability: each method is rerun on even and odd 60-s blocks and the two labelings
compared. This is the **noise ceiling**: membership will look different between any two stretches of data by this
much even when nothing has changed, so a later "neurons joined or left" result has to exceed it. Agreement
between methods: adjusted Rand index (ARI; 0 = chance, 1 = identical), with a label-permutation null.

In [1]:
import sys, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import awake_grounding as ag
OUT = REPO / "data" / "out" / "2026-11-15_ensembles_awake"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
O = [ag.load(f) for f in sorted(glob.glob(str(REPO / "data/processed/ensembles_awake/*.pkl")))]
META = pd.DataFrame([o["meta"] for o in O]); UNITS = pd.concat([o["units"] for o in O], ignore_index=True)
REL = pd.concat([o["reliability"] for o in O], ignore_index=True); AGR = pd.concat([o["agreement"] for o in O], ignore_index=True)
DUP = pd.concat([o["duplicates"] for o in O if len(o["duplicates"])], ignore_index=True)
META.to_csv(OUT / "signals_qc.csv", index=False); UNITS.to_csv(OUT / "units_descriptors.csv", index=False)
REL.to_csv(OUT / "reliability.csv", index=False); AGR.to_csv(OUT / "agreement.csv", index=False); DUP.to_csv(OUT / "duplicates.csv", index=False)
METHODS = ["lfp original M1 (raw)", "lfp original M1 (population removed)", "lfp tracker expression M1 (raw)",
           "lfp tracker expression M1 (population removed)", "getclust", "corr"]
SHORT = {"lfp original M1 (raw)": "LFP original\n(raw)", "lfp original M1 (population removed)": "LFP original\n(pop. removed)",
         "lfp tracker expression M1 (raw)": "LFP tracker\n(raw)", "lfp tracker expression M1 (population removed)": "LFP tracker\n(pop. removed)",
         "getclust": "getClust", "corr": "correlation\ncommunities"}
COL = {"lfp original M1 (raw)": "#2a78d6", "lfp original M1 (population removed)": "#86b6ef", "lfp tracker expression M1 (raw)": "#1baf7a",
       "lfp tracker expression M1 (population removed)": "#8fd9bd", "getclust": "#eb6834", "corr": "#6250d6"}
print(META[["n_good", "n_low_rate", "n_duplicates", "n_kept"]].sum().to_dict())
UNITS[["rate_hz", "cv2", "fano", "pop_coupling", "waveform_ms"]].describe().round(2)

{'n_good': 2629, 'n_low_rate': 195, 'n_duplicates': 11, 'n_kept': 2429}


,rate_hz,cv2,fano,pop_coupling,waveform_ms
count,2429.00,2429.00,2429.00,2429.00,2429.00
mean,7.73,0.91,3.84,0.24,0.57
std,8.39,0.19,3.80,0.34,0.18
min,0.50,0.26,0.23,-0.85,0.12
25%,2.35,0.79,1.98,0.01,0.48
50%,5.40,0.90,2.93,0.25,0.58
75%,10.12,1.03,4.44,0.51,0.67
max,159.08,1.65,60.44,0.96,2.20


## 1. How many neurons does each method assign, and how reliable is it?
Left: share of neurons assigned to any ensemble. Middle: split-half ARI over all neurons (in/out and which group).
Right: split-half agreement on neurons assigned in **both** halves (which group / which pole).

In [2]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
rng = np.random.default_rng(0)
for ax, col, lab in zip(axes, ["share_assigned", "half_ari", "half_ari_assigned"],
                        ["share of neurons assigned", "split-half ARI (all neurons)", "split-half ARI (assigned in both halves)"]):
    for k, mth in enumerate(METHODS):
        v = REL[REL.method == mth][col].dropna()
        ax.scatter(k + rng.uniform(-0.2, 0.2, len(v)), v, s=12, color=COL[mth], alpha=0.85)
        ax.plot([k - 0.3, k + 0.3], [v.median()] * 2, color="0.1", lw=2)
        ax.text(k, 1.06, f"{v.median():.2f}", ha="center", fontsize=8)
    ax.set_ylim(-0.25, 1.12); ax.axhline(0, color="0.75", lw=0.6); ax.set_title(lab, fontsize=9, loc="left")
    ax.set_xticks(range(len(METHODS)), [SHORT[m] for m in METHODS], fontsize=7.5)
fig.suptitle("Ensemble definitions: coverage and split-half reliability (bar and number = median over signals)", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "1_coverage_reliability"); plt.show()
S1 = REL.groupby("method").agg(signals=("half_ari", "size"), share_assigned=("share_assigned", "median"), groups=("n_groups", "median"),
                               half_ari=("half_ari", "median"), half_ari_assigned=("half_ari_assigned", "median"),
                               half_kappa_in_out=("half_kappa_assigned", "median")).reindex(METHODS)
S1.to_csv(OUT / "1_reliability_summary.csv"); S1.round(2)

,signals,share_assigned,groups,half_ari,half_ari_assigned,half_kappa_in_out
method,,,,,,
lfp original M1 (raw),61,0.60,2.0,0.43,1.00,0.53
lfp original M1 (population removed),61,0.50,2.0,0.27,1.00,0.32
lfp tracker expression M1 (raw),42,0.62,2.0,0.35,1.00,0.49
lfp tracker expression M1 (population removed),42,0.63,2.0,0.35,1.00,0.42
getclust,65,0.71,2.0,0.29,0.51,0.34
corr,65,0.62,3.0,0.39,0.70,0.45


## 2. Do the methods agree with each other?
Median ARI between every pair of methods (all neurons); stars: share of signals where the pair beats its
label-permutation null.

In [3]:
idx = {m: i for i, m in enumerate(METHODS)}
Mx = np.full((len(METHODS), len(METHODS)), np.nan); Sg = np.full_like(Mx, np.nan)
for (a, b), d in AGR.groupby(["a", "b"]):
    Mx[idx[a], idx[b]] = Mx[idx[b], idx[a]] = d.ari.median()
    Sg[idx[a], idx[b]] = Sg[idx[b], idx[a]] = (d.p < .05).mean()
fig, ax = plt.subplots(figsize=(7, 5.6))
im = ax.imshow(Mx, cmap="Blues", vmin=0, vmax=0.7)
for i in range(len(METHODS)):
    for j in range(len(METHODS)):
        if i != j and np.isfinite(Mx[i, j]):
            ax.text(j, i, f"{Mx[i, j]:.2f}\n{Sg[i, j]:.0%}", ha="center", va="center", fontsize=7.5,
                    color="white" if Mx[i, j] > 0.4 else "0.15")
ax.set_xticks(range(len(METHODS)), [SHORT[m] for m in METHODS], fontsize=7.5, rotation=0)
ax.set_yticks(range(len(METHODS)), [SHORT[m].replace("\n", " ") for m in METHODS], fontsize=7.5)
fig.colorbar(im, ax=ax, fraction=0.04, label="median ARI")
ax.set_title("Agreement between methods (median ARI; % of signals beating the permutation null)", fontsize=9, loc="left")
save(fig, "2_agreement"); plt.show()

## 3. Who gets assigned? Firing descriptors of assigned vs unassigned neurons
Per signal, median of each descriptor among assigned and unassigned neurons; paired over signals.

In [4]:
rows = []
for o in O:
    L = o["labels"]; u = o["units"].set_index("unit_id").loc[L.unit_id]
    for mth in ["lfp original M1 (raw)", "lfp original M1 (population removed)", "getclust", "corr"]:
        if mth not in L: continue
        a = L[mth].to_numpy() != 0
        if a.sum() < 2 or (~a).sum() < 2: continue
        for dsc in ["rate_hz", "pop_coupling", "cv2", "narrow"]:
            f = (lambda v: v.mean()) if dsc == "narrow" else (lambda v: v.median())
            rows.append({"method": mth, "descriptor": dsc, "assigned": f(u[dsc][a].astype(float)), "unassigned": f(u[dsc][~a].astype(float))})
DS = pd.DataFrame(rows)
S3 = DS.groupby(["method", "descriptor"]).apply(lambda d: pd.Series({
    "signals": len(d), "assigned (median)": d.assigned.median(), "unassigned (median)": d.unassigned.median(),
    "assigned higher": f"{(d.assigned > d.unassigned).sum()}/{len(d)}", "p": wilcoxon(d.assigned, d.unassigned).pvalue})).reset_index()
S3.to_csv(OUT / "3_assigned_vs_unassigned.csv", index=False)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, dsc, lab in zip(axes, ["rate_hz", "pop_coupling", "cv2", "narrow"], ["rate (Hz)", "population coupling", "CV2 (burstiness)", "share narrow-spiking"]):
    for k, mth in enumerate(["lfp original M1 (raw)", "lfp original M1 (population removed)", "getclust", "corr"]):
        d = DS[(DS.method == mth) & (DS.descriptor == dsc)]
        for row in d.itertuples(): ax.plot([k - 0.15, k + 0.15], [row.unassigned, row.assigned], color="0.85", lw=0.6)
        ax.scatter(np.full(len(d), k - 0.15), d.unassigned, s=8, color="0.6")
        ax.scatter(np.full(len(d), k + 0.15), d.assigned, s=8, color=COL[mth])
    ax.set_xticks(range(4), ["LFP orig.\n(raw)", "LFP orig.\n(pop. rm)", "getClust", "corr."], fontsize=7.5)
    ax.set_title(lab, fontsize=9, loc="left")
    if dsc == "rate_hz": ax.set_yscale("log")
fig.suptitle("Unassigned (grey, left) vs assigned (colour, right), one line per signal", fontsize=10, x=0.01, ha="left")
fig.tight_layout(); save(fig, "3_assigned_vs_unassigned"); plt.show()
S3.round(3)

,method,descriptor,signals,assigned (median),unassigned (median),assigned higher,p
0,corr,cv2,56,0.869,0.932,14/56,0.000
1,corr,narrow,56,0.202,0.046,40/56,0.000
2,corr,pop_coupling,56,0.346,0.201,40/56,0.000
3,corr,rate_hz,56,5.530,4.450,40/56,0.001
4,getclust,cv2,63,0.904,0.910,27/63,0.262
5,getclust,narrow,63,0.176,0.045,41/63,0.000
6,getclust,pop_coupling,63,0.366,0.066,55/63,0.000
7,getclust,rate_hz,63,5.140,4.437,35/63,0.385
8,lfp original M1 (population removed),cv2,57,0.897,0.910,24/57,0.307
9,lfp original M1 (population removed),narrow,57,0.158,0.118,33/57,0.115


## Conclusions
- **Units.** 2,629 well-isolated units → 2,429 kept (195 below 0.5 Hz awake, 11 duplicates). Median 5.4 Hz; 15 %
  narrow-spiking.
- **Not every neuron is in opponency.** With per-neuron significance, a median 60 % of neurons couple to the
  original motif (50 % with the population mean removed; range 0–95 % across signals); getClust assigns 71 %,
  correlation communities 62 %.
- **Which pole is very reliable; in-or-out is not.** Neurons assigned in both halves never switch pole (split-half
  ARI 1.00 for every LFP definition), but whether a neuron is assigned at all agrees only moderately between halves
  (kappa 0.32–0.53). That is the noise ceiling for the user's hypothesis: within one awake state, about half of
  the in/out calls differ between two halves of the data. "Neurons joining or leaving with the baseline" has to
  beat this.
- **The definitions agree above chance but are not interchangeable.** LFP definitions agree with each other
  (ARI 0.4–0.7). LFP-independent methods agree with the LFP split weakly (ARI 0.08–0.21, beating chance in about
  half to two thirds of signals); among neurons both methods assign, agreement is much higher (0.5–0.76), so they
  mostly disagree on who is in, not on how members are split.
- **Assigned neurons differ in firing.** They fire faster (5.5 vs 4.0 Hz for LFP coupling), track the population
  more, and include more narrow-spiking (putative interneuron) cells (18–20 % vs 4–13 %). Higher rates give more
  statistical power, so part of in/out is detectability: the isoflurane analysis must rate-match or model rate
  before reading membership changes as biology.